# Hospitality PMS LLM — Evaluation Harness (Colab)

Runs the full evaluation matrix on a free T4 GPU.

**Before running:**
1. Runtime → Change runtime type → T4 GPU
2. Upload `vectorstore.zip` and `benchmark_dev.jsonl` when prompted

**To prepare locally:**
```bash
cd hospitality-pms-llm
cd output && zip -r vectorstore.zip vectorstore/ && cd ..
```

In [ ]:
# 1. Install dependencies
!pip install -q llama-cpp-python \
    chromadb==1.5.9 \
    sentence-transformers==5.6.0 \
    huggingface_hub

In [ ]:
# 2. Download models from HuggingFace (fast on Colab network)
!huggingface-cli download Qwen/Qwen2.5-3B-Instruct-GGUF qwen2.5-3b-instruct-q4_k_m.gguf --local-dir ./models
!huggingface-cli download Qwen/Qwen2.5-7B-Instruct-GGUF \
    qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf \
    qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf \
    --local-dir ./models

In [ ]:
# 3. Upload vectorstore.zip and benchmark_dev.jsonl
from google.colab import files
import os, shutil

print("Upload vectorstore.zip and benchmark_dev.jsonl")
uploaded = files.upload()

if 'vectorstore.zip' in uploaded:
    os.makedirs('output', exist_ok=True)
    shutil.move('vectorstore.zip', 'output/vectorstore.zip')
    !cd output && unzip -qo vectorstore.zip && rm vectorstore.zip
    print("Vectorstore extracted.")

if 'benchmark_dev.jsonl' in uploaded:
    os.makedirs('data', exist_ok=True)
    shutil.move('benchmark_dev.jsonl', 'data/benchmark_dev.jsonl')
    print("Benchmark loaded.")

In [ ]:
# 4. Verify setup
import os
print("Models:")
for f in os.listdir('models'):
    if f.endswith('.gguf'):
        size_gb = os.path.getsize(f'models/{f}') / 1e9
        print(f"  {f} ({size_gb:.1f} GB)")

print(f"\nVectorstore exists: {os.path.exists('output/vectorstore')}")
print(f"Benchmark exists: {os.path.exists('data/benchmark_dev.jsonl')}")

import json
with open('data/benchmark_dev.jsonl') as f:
    tasks = [json.loads(l) for l in f]
print(f"Benchmark tasks: {len(tasks)}")

In [ ]:
# 5. Pipeline + eval code (self-contained, no imports from repo)

import json
import time
import os
from datetime import datetime
from dataclasses import dataclass, field

import chromadb
from sentence_transformers import SentenceTransformer

VECTORSTORE_DIR = 'output/vectorstore'
MODELS_DIR = 'models'
EMBEDDING_MODEL = 'BAAI/bge-small-en-v1.5'
COLLECTION_NAME = 'hospitality_pms'

SYSTEM_PROMPT = """You are a hospitality technology expert specializing in Oracle OPERA Cloud PMS \
and the Oracle Hospitality Integration Platform (OHIP). You help hotel IT teams, \
system integrators, and developers with API orchestration, system configuration, \
and troubleshooting."""

SYSTEM_PROMPT_RAG = SYSTEM_PROMPT + """

Answer based on the provided context. If the context doesn't contain enough \
information, say so rather than guessing. When referencing API endpoints, \
include the HTTP method and full path. When referencing configuration, \
specify the exact OPERA Control or setting name."""

SYSTEM_PROMPT_NO_RAG = SYSTEM_PROMPT + """

Answer from your training knowledge. When referencing API endpoints, \
include the HTTP method and full path where possible. When referencing \
configuration, specify the exact setting name if you know it."""

EVAL_CONFIGS = {
    '3B-base':  {'backend': 'llama_cpp', 'model_name': 'qwen2.5-3b-instruct-q4_k_m.gguf', 'use_rag': False},
    '3B-RAG':   {'backend': 'llama_cpp', 'model_name': 'qwen2.5-3b-instruct-q4_k_m.gguf', 'use_rag': True},
    '7B-base':  {'backend': 'llama_cpp', 'model_name': 'qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf', 'use_rag': False},
    '7B-RAG':   {'backend': 'llama_cpp', 'model_name': 'qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf', 'use_rag': True},
}


@dataclass
class RetrievedChunk:
    text: str
    metadata: dict
    distance: float


class RAGPipeline:
    def __init__(self, backend, model_name, use_rag, top_k=5):
        self.backend = backend
        self.model_name = model_name
        self.use_rag = use_rag
        self.top_k = top_k

        if use_rag:
            self._chroma = chromadb.PersistentClient(path=VECTORSTORE_DIR)
            self._collection = self._chroma.get_collection(COLLECTION_NAME)
            self._embedder = SentenceTransformer(EMBEDDING_MODEL)
        else:
            self._chroma = None
            self._collection = None
            self._embedder = None

        from llama_cpp import Llama
        model_path = os.path.join(MODELS_DIR, model_name)
        self._llm = Llama(
            model_path=model_path,
            n_ctx=16384,
            n_gpu_layers=-1,
            verbose=False,
        )

    def retrieve(self, query):
        if not self.use_rag:
            return []
        qe = self._embedder.encode(query, normalize_embeddings=True).tolist()
        results = self._collection.query(
            query_embeddings=[qe], n_results=self.top_k,
            include=['documents', 'metadatas', 'distances'],
        )
        return [
            RetrievedChunk(text=d, metadata=m, distance=dist)
            for d, m, dist in zip(
                results['documents'][0],
                results['metadatas'][0],
                results['distances'][0],
            )
        ]

    def query(self, question):
        chunks = self.retrieve(question)
        if chunks:
            ctx_parts = []
            for i, c in enumerate(chunks, 1):
                src = c.metadata.get('source_file', '')
                sec = c.metadata.get('section', c.metadata.get('path', ''))
                ctx_parts.append(f'[Source {i}: {src} — {sec}]\n{c.text}')
            context = '\n\n---\n\n'.join(ctx_parts)
            user_prompt = f'Context:\n{context}\n\nQuestion: {question}'
            system = SYSTEM_PROMPT_RAG
        else:
            user_prompt = question
            system = SYSTEM_PROMPT_NO_RAG

        response = self._llm.create_chat_completion(
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user', 'content': user_prompt},
            ],
            max_tokens=2048,
        )
        answer = response['choices'][0]['message']['content']
        return answer, chunks


def run_eval(config_name, tasks):
    print(f'\n{"="*60}')
    print(f'Config: {config_name}')
    print(f'{"="*60}')

    cfg = EVAL_CONFIGS[config_name]
    pipeline = RAGPipeline(**cfg)
    results = []

    for i, task in enumerate(tasks, 1):
        q = task['question']
        print(f'  [{i}/{len(tasks)}] {task["id"]}: {q[:70]}...')

        start = time.time()
        answer, chunks = pipeline.query(q)
        elapsed = time.time() - start

        results.append({
            'task_id': task['id'],
            'config': config_name,
            'model': cfg['model_name'],
            'use_rag': cfg['use_rag'],
            'question': q,
            'expected_answer': task.get('expected_answer', ''),
            'generated_answer': answer,
            'category': task.get('category', ''),
            'module': task.get('module', ''),
            'difficulty': task.get('difficulty', ''),
            'num_chunks_retrieved': len(chunks),
            'chunk_distances': [c.distance for c in chunks],
            'latency_seconds': round(elapsed, 2),
        })
        print(f'         {elapsed:.1f}s | {len(answer)} chars')

    del pipeline
    import gc; gc.collect()
    return results

In [ ]:
# 6. Run evaluation — all 4 local configs

with open('data/benchmark_dev.jsonl') as f:
    tasks = [json.loads(l) for l in f]
print(f'Loaded {len(tasks)} tasks')

configs_to_run = ['3B-base', '3B-RAG', '7B-base', '7B-RAG']
all_results = []

for config_name in configs_to_run:
    try:
        results = run_eval(config_name, tasks)
        all_results.extend(results)
    except Exception as e:
        print(f'ERROR running {config_name}: {e}')
        import traceback; traceback.print_exc()

# Save results
os.makedirs('output/eval_results', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = f'output/eval_results/eval_dev_{ts}.jsonl'

with open(out_path, 'w') as f:
    for r in all_results:
        f.write(json.dumps(r) + '\n')

print(f'\nResults: {len(all_results)} total → {out_path}')

# Quick summary
from collections import defaultdict
by_config = defaultdict(list)
for r in all_results:
    by_config[r['config']].append(r['latency_seconds'])

print(f'\n{"Config":<15} {"Tasks":>6} {"Avg Latency":>12}')
print('-' * 35)
for cfg in configs_to_run:
    lats = by_config[cfg]
    if lats:
        print(f'{cfg:<15} {len(lats):>6} {sum(lats)/len(lats):>10.1f}s')

In [ ]:
# 7. Download results
from google.colab import files
files.download(out_path)
print(f'Downloaded {out_path}')